# Ablación del pipeline de entrada – TartuNLP-ru → dev SV

**Pregunta:** ¿mejora el modelo actual (sin reentrenar) si le damos la entrada limpia?

| Variante | Entrada |
|---|---|
| `none` | datos tal cual (= resultado actual: F1 macro 0.443, AUC 0.58) |
| `core` | fix_span + dehyphen + detok + ws |
| `core_mark` | core + palabra marcada `<t> </t>` |

Ojo con `core_mark`: el adaptador ruso **no** se entrenó con marcas, así que aquí puede empeorar. Las marcas son para M1 (reentrenar); esta fila solo mide cuánto molestan sin entrenar.

**Necesita GPU** (*Entorno de ejecución → Cambiar tipo → GPU T4*). Unos minutos por variante.

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
DATA_DIR = f'{DRIVE}/semeval27/dev'                       # SV/, ... con usages.jsonl
ADAPTER  = f'{DRIVE}/axolotl_models/ru/classification'    # adaptador TartuNLP-ru
PREDS    = f'{DRIVE}/semeval27/preds'
RESULTS  = f'{DRIVE}/semeval27/results'
!mkdir -p {PREDS} {RESULTS}

## 2. Repo (clonar la primera vez, `pull` las siguientes)
Si el repo es privado: guarda un token de GitHub en los *Secrets* de Colab (icono de la llave) con el nombre `GITHUB_TOKEN`.

In [ ]:
import os
REPO_DIR = '/content/repo'
url = 'https://github.com/tonyOehme/hcc_practical_course_semeval2027.git'
try:
    from google.colab import userdata
    url = url.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass  # repo público o sin token

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {url} {REPO_DIR}
%cd {REPO_DIR}/task2/semeval27_pipeline
!git log --oneline -1

## 3. Entorno
Crea `/content/venv` (Python 3.11 + versiones de TartuNLP) e instala el paquete `semeval27`. La primera vez tarda unos minutos; después es instantáneo.
Para análisis solo en CPU basta con `--core`.

In [ ]:
!bash scripts/colab_setup.sh
PY = '/content/venv/bin/python'   # para los scripts (entorno TartuNLP)

# Para usar semeval27 también en las celdas de este notebook (kernel de Colab):
import sys
sys.path.insert(0, '/content/repo/task2/semeval27_pipeline')

## 4. Convertir y predecir las tres variantes
Si una predicción ya está en Drive, no se repite.

In [ ]:
import os
VARIANTS = {'none': '', 'core': '--steps core', 'core_mark': '--steps core --mark_target'}
for tag, flags in VARIANTS.items():
    !{PY} scripts/convert_data.py --data_dir {DATA_DIR} --out_dir outputs/pairs {flags} --tag {tag}
    out = f'{PREDS}/SV.tartunlp_ru.{tag}.tsv'
    if os.path.exists(out):
        print('ya existe:', out); continue
    !{PY} scripts/predict_tartunlp.py --pairs outputs/pairs/SV.{tag}.pairs.tsv --adapter_dir {ADAPTER} --out {out}

## 5. Comparar
- **F1 @0.35**: umbral fijo de TartuNLP (lo que se enviaría).
- **AUC medio por palabra**: ¿ordena mejor los usos? Es la métrica que nos dice si la entrada ayuda al *modelo*, independientemente del umbral.
- Referencia: baseline "siempre 1" = 0.472 (macro) / 0.525 (global).

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from semeval27.metrics import summary

rows = []
for tag in VARIANTS:
    df = pd.read_csv(f'{PREDS}/SV.tartunlp_ru.{tag}.tsv', sep='\t', dtype={'sentence_id': str})
    auc = df.groupby('word')[['label', 'prob']].apply(lambda g: roc_auc_score(g['label'], g['prob']))
    s = summary(df.assign(pred=(df['prob'] >= 0.35).astype(int)))
    rows.append({'variante': tag, 'F1 macro @0.35': s['f1_macro_words'], 'F1 global @0.35': s['f1_global'],
                 'AUC medio': round(auc.mean(), 3), 'pred_rate': round((df['prob'] >= 0.35).mean(), 3),
                 **{f'AUC {w}': round(a, 2) for w, a in auc.items()}})
res = pd.DataFrame(rows).set_index('variante')
res.to_csv(f'{RESULTS}/ablation_input_SV.tsv', sep='\t')
res.T

## Conclusión (2026-11-23, Colab T4)
- AUC medio: none 0.583 · core 0.575 · core_mark 0.567 → **la entrada limpia no mejora el modelo** (diferencias ≈ ruido, ±0.04 por palabra).
- El F1 sube un poco solo porque sube el pred_rate (0.87 → 0.92): se acerca a "siempre 1", no acierta más.
- Excepción: *motiv* 0.61 → 0.65, la única palabra cuya definición cambió (guiones blandos).
- Marcar la palabra sin reentrenar no ayuda (era lo esperado).
- → El problema es el modelo (H1/H3): siguiente paso, M1.